In [1]:
from experiment import Experiment
from database import PatchDataset
from models import LinearModel, SWENetV0
from metrics import WeightedMSE
from optimizer import AdamOptimizer
import pandas as pd

In [ ]:
ds = PatchDataset(
    input_variables=[
        'inm_snow_cover', 'inm_swe', 'inm_t2m', 'inm_olr', 'inm_ww', 'inm_ts', 'inm_v850', 'inm_u850', 'inm_tp', 'inm_h500', 'inm_hlt', 'inm_mslp',
        'inm_cos_lat', 'inm_sin_lon', 'inm_cos_lon', 'inm_lead_time', 'inm_cos_day', 'inm_sin_day', 'inm_year_norm',
        'snow_cover', 'sd', 't2m', 'tp', 'pt', 'sst', 'lsm', 'glaicer', 'z', 'sdor',
        'cos_lat', 'sin_lon', 'cos_lon', 'cos_day', 'sin_day', 'year_norm'
    ],
    target_variables=['sd', 'lat', 'lon', 'day', 'year', 'inm_lead_time', 'sd_clim', 'glaicer'],
    modes={
        'train': {"t_min": '19910101', 't_max': '20201231', 'epoch_size': 500, 'batch_size': 8},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 100, 'batch_size': 8},
    },
    era_scales=[{'id': 'local', 'xSize': 64, 'ySize': 64, 'tSize': 7, 'xyStep': 1, 'tStep': 1},
                {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 7},
                {'id': 'global', 'xSize': 88, 'ySize': 16, 'tSize': 6, 'xyStep': 16, 'tStep': 30, 'fixY': 4}],
    inm_scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
                {'id': 'global', 'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 4, 'tStep': 7, 'fixY': 4}],
    target_scale={'xSize': 64, 'ySize': 64, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
    mask='snow',
    with_climate=['t2m', 'ts', 'sst', 'tp', 'swe', 'sd'],
    num_workers=8
)

In [2]:
ds = PatchDataset(
    input_variables=[
        'inm_swe', 'inm_t2m', 'inm_tp',
        'inm_cos_lat', 'inm_sin_lon', 'inm_cos_lon', 'inm_lead_time', 'inm_cos_day', 'inm_sin_day', 'inm_year_norm',
        'sd', 't2m', 'tp', 'sst', 'lsm', 'z', 'sdor',
        'cos_lat', 'sin_lon', 'cos_lon', 'cos_day', 'sin_day', 'year_norm'
    ],
    target_variables=['sd', 'lat', 'lon', 'day', 'year', 'inm_lead_time', 'sd_clim', 'glaicer'],
    modes={
        'train': {"t_min": '19910101', 't_max': '20201231', 'epoch_size': 500, 'batch_size': 8},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 100, 'batch_size': 8},
    },
    era_scales=[{'id': 'local', 'xSize': 64, 'ySize': 64, 'tSize': 7, 'xyStep': 1, 'tStep': 1},
                {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 7},
                {'id': 'global', 'xSize': 88, 'ySize': 16, 'tSize': 6, 'xyStep': 16, 'tStep': 30, 'fixY': 4}],
    inm_scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
                {'id': 'global', 'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 4, 'tStep': 7, 'fixY': 4}],
    target_scale={'xSize': 64, 'ySize': 64, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
    mask='snow',
    with_climate=['t2m', 'ts', 'sst', 'tp', 'swe', 'sd'],
    num_workers=8
)

In [3]:
model = LinearModel('inm_swe_regional', size=(64, 64))
metric = WeightedMSE('sd', mask=[('glaicer', '<', 0.5), ('sd_clim', '>', 0)])
optimizer = AdamOptimizer(lr=0.001)
experiment = Experiment(f'swe_base', ds, model, metric, optimizer, modes=['train', 'test'])
experiment.run(1)

train epoch   1:   0%|          | 0/500 [00:00<?, ?batch/s]

test  epoch   1:   0%|          | 0/100 [00:00<?, ?batch/s]

In [3]:
spatial = ['cos_lat', 'cos_lon', 'sin_lon']
temporal = ['cos_day', 'sin_day', 'year_norm']
inm_temporal = temporal + ['lead_time']
num_stats = 1
channels = 7 + len(spatial) + len(temporal)
inm_channels = num_stats*3 + len(spatial) + len(inm_temporal)

model = SWENetV0({},
    scales={
        'local':        {'channels': channels, 'spatial': spatial, 'temporal': temporal},
        'regional':     {'channels': channels, 'spatial': spatial, 'temporal': temporal},
        'global':       {'channels': channels, 'spatial': spatial, 'temporal': temporal},
        'inm_regional': {'channels': inm_channels, 'spatial': spatial, 'temporal': inm_temporal},
        'inm_global':   {'channels': inm_channels, 'spatial': spatial, 'temporal': inm_temporal},
})
metric = WeightedMSE('sd', mask=[('glaicer', '<', 0.5), ('sd_clim', '>', 0)])
optimizer = AdamOptimizer(lr=0.001)
experiment = Experiment(f'swe_net_v0', ds, model, metric, optimizer, modes=['train', 'test'])
experiment.run(1)

train epoch   1:   0%|          | 0/500 [00:00<?, ?batch/s]

cos_lat torch.Size([8, 64])
cos_lon torch.Size([8, 64])
sin_lon torch.Size([8, 64])
torch.Size([32768, 7, 13])
cos_lat torch.Size([8, 16])
cos_lon torch.Size([8, 16])
sin_lon torch.Size([8, 16])
torch.Size([2048, 8, 13])
cos_lat torch.Size([8, 16])
cos_lon torch.Size([8, 88])
sin_lon torch.Size([8, 88])
torch.Size([11264, 6, 13])
cos_lat torch.Size([8, 16])
cos_lon torch.Size([8, 16])
sin_lon torch.Size([8, 16])
torch.Size([2048, 28, 10])
cos_lat torch.Size([8, 16])
cos_lon torch.Size([8, 16])
sin_lon torch.Size([8, 16])
torch.Size([2048, 4, 10])


TypeError: unsupported operand type(s) for -: 'NoneType' and 'Tensor'